## Data Distribution & Consistency
Hasta este punto, ya sabemos si faltan datos o existen duplicados; aohra queremos conocer qué valores contienen las variables numéricas, cómo se distribuyen y si encontramos cantidades que ameriten investigación.

**Objetivo**. Identifciar rangos, valores extremos y posibles inconsistencias sin corregirlos todavía.



In [0]:
import pandas as pd
from pathlib import Path # Para trabajar con rutas de archivos de manera más organizada


# Contrucción de rutas de datasets y validación de archivos exists
DATA_PATH = Path('../data/raw')

# Lectura de dataset
sales = DATA_PATH / 'Sales_dataset.csv'
print("Sales file exists:", sales.exists())

inventory = DATA_PATH / 'Inventory_dataset.csv'
print("Inventory file exists:", inventory.exists())

# Carga de datos
dfSales = pd.read_csv(sales, dtype={"cd_NumParte": "string"})

dfInventory = pd.read_csv(inventory, dtype= {"cd_NumParte": "string"})

In [0]:
# Identifciar columnas numéricas
sales_numeric_columns = dfSales.select_dtypes(
    include ="number"
).columns.tolist()

inventory_numeric_columns = dfInventory.select_dtypes(
    include ="number"
).columns.tolist()

print("Columnas numéricas de Sales")
print(sales_numeric_columns)

print("\nColumnas numéricas de Inventory")
print(inventory_numeric_columns)

In [0]:
#Estadística descriptivade Sales

display(
    dfSales[sales_numeric_columns].describe() #.T, transpone el resultado
)

In [0]:
#Estadística descriptiva de Inventory

display(
    dfInventory[inventory_numeric_columns].describe().T #T, transpone el resultado
)

In [0]:
# Seleccionar variables relevantes, no todas las columnas numéricas nos interesan igual.
sales_analysis_columns = [
    "ct_Cantidad",
    "im_PrecioVenta"
]

display(
    dfSales[sales_analysis_columns].describe().T
)

# Para inventario, primero revisar los nombres reales
print(dfInventory.columns.to_list())

inventory_analysis_columns = [
    "ct_Existencia",
    "ct_Reservado",
    "nu_PuntoReorden",
    "im_ValorInventario",
    "ct_VentasUltimos12Meses",
    "ct_DiasEnInventario"
]

display(
    dfInventory[inventory_analysis_columns].describe().T)

In [0]:
#Cantidades negativas, cero y positivas
negative_sales = dfSales[
    dfSales["ct_Cantidad"] < 0
]

zero_sales = dfSales[
    dfSales["ct_Cantidad"] == 0
]

positive_sales = dfSales[
    dfSales["ct_Cantidad"] > 0
]
# Cantidad de ventas negativas, cero y positivas
print("Ventas negativas:", len(negative_sales))
print("Ventas en cero:", len(zero_sales))
print("Ventas positivas:", len(positive_sales))

con el análisis anterior tenemos 3 categorías:
- < 0 -> investigar
- = 0 -> investigar
- > 0 -> comportamiento esperado inicialmente

No concluimos todavía que negativo = incorrecto, porque una cantidad egativa puede representar una devolución, una cancelación o algún movimiento permitido por el sistema.

Primero conocemos la regla de negocio.

In [0]:
# Validar precios
invalid_prices = dfSales[
    dfSales["im_PrecioVenta"] <= 0
]

print("Precios inválidos:", len(invalid_prices))

#display(invalid_prices.head(10)) #Empty dataset

##Pregunta de Negocio
¿Puede existir legítimamente una refacción con precio de venta igual a cero?, esto genera un registro para revisión.

##IQR: Interquartile Range

In [0]:
#Detectar posibles outliers
#IQR: Interquartile Range

Q1 = dfSales["ct_Cantidad"].quantile(0.25)
Q3 = dfSales["ct_Cantidad"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", lower_limit)
print("Límite superior:", upper_limit)

La lógica es:
**IQR = Q3 - Q1**

y usamos como regla exploratoria:
Límite inferior = Q1 - 1.5(IQR)
Límite superior = Q3 + 1.5(IQR)

In [0]:
#Ver los posibles outliers
quantity_outliers = dfSales [
    (dfSales["ct_Cantidad"] < lower_limit) | 
    (dfSales["ct_Cantidad"] > upper_limit)
]

print(
    "Posibles outliers de cantidad:",
    len(quantity_outliers)
)

"""display(
    quantity_outliers[
        [
            "cd_NumParte",
            "ct_Cantidad",
            "im_PrecioVenta"
        ]
    ].head(20)
)
"""

##Muy importante: OUtlier != Error

Un cliente normalmente compra: 1,4,6,2,4 productos y un día compra 50, esto puede aparecer como outlier, pero quizas fue:
- compra de flotilla
- promoción
- pedido mayorista
- temporada especial

Entonces se deberá considerar como **valor atípico** y no **dato incorrecto**

## Distribución de cantidad (primer gráfica sencilla)

In [0]:
#Gráfica de Data Profiling
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

dfSales["ct_Cantidad"].hist(bins=30)

plt.title("Distribución de cantidad vendida")
plt.xlabel("Cantidad")
plt.ylabel("Frecuencia")

plt.show()



In [0]:
#Boxplot de cantidad
plt.figure(figsize=(10, 4))

plt.boxplot(
    dfSales["ct_Cantidad"].dropna(),
    vert=False
)

plt.title("Distribución de cantidad vendida")
plt.xlabel("Cantidad")

plt.show()

Aquí dropna() si tiene sentido porque no estamos borrando registros del dataset, solo le decimos al gráfico que ignore los valores faltantes.

##Revisar inventario disponible

**Variable Derivada**

In [0]:
#Localizar nombres exactos de existencia y reservado 
dfInventory.columns.to_list()

dfInventory["ct_Disponible"] = (
    dfInventory["ct_Existencia"]
    - dfInventory["ct_Reservado"]
)

display(
    dfInventory[
        [
            "ct_Existencia",
            "ct_Reservado",
            "ct_Disponible"
        ]
    ].describe().T
)

**Buscar disponibilidad negativa**

In [0]:
negative_inventory = dfInventory[
    dfInventory["ct_Disponible"] < 0
]

print(
    "Productos con disponibilidad negativa:",
    len(negative_inventory)
)

display(
    negative_inventory.head(10)
)

**Primera aproximación a movimiento**

Tengo inventario... ¿pero ese inventario realmente se mueve?

In [0]:
#Si el inventario contiene efectivamente la variable de ventas de los últimos 12 meses, podemos hacer:

no_movement = dfInventory[
    dfInventory["ct_VentasUltimos12Meses"] == 0
]

print(
    "Productos sin ventas en los últimos 12 meses:",
    len(no_movement)
)

#Inventario sin movimiento
inventory_no_movement = dfInventory[
    (dfInventory["ct_Existencia"] > 0)
    &
    (dfInventory["ct_VentasUltimos12Meses"] == 0)
]

print(
    "Productos con inventario pero sin movimiento:",
    len(inventory_no_movement)
)



**¿Dónde tengo inventario que no se está moviendo?**

##Observaciones

**Conclusiones de distribución y consistencia**
Después de revisar la distribución de las principales varibales observo que:

1. La cantidad vendida presenta un **rango entre -12.5 y 31.5** unidades.
2. Se identificaron **47 transacciones** con cantidad igual a **cero**.
3. Se identificaron **0** transacciones con **cantidad negativa**.
4. El análisis mediante IQR identificó **0* posibles valores **atípicos**.
5. La existencia disponible presenta **valores entre 20 y -20 **unidades.
6. Se identificaron **461** productos con **inventario disponible negativo**.
7. Se identificaron **265** productos **con existencia pero sin ventas durante los últimos 12 mese**s.


## Ejercicio 

**1. ¿Por qué un outlier no debe eliminarse automáticamente?**

Porque un valor atípico no es necesariamente un dato incorrecto: puede reflejar un comportamiento real del negocio (compra de flotilla, promoción, pedido mayorista, temporada especial). Eliminarlo automáticamente destruiría información legítima. Antes de decidir hay que investigar la causa: si es un error de captura se corrige o elimina con justificación; si es un evento real, se conserva (y en algunos casos hasta se analiza por separado, porque los clientes grandes suelen explicar buena parte de la demanda). Outlier ≠ Error.

**2. ¿Qué diferencia existe entre ct_Existencia y ct_Disponible?**

'ct_Existencia' es el stock físico total registrado en almacén. 'ct_Disponible' es una **variable derivada**: existencia menos lo ya reservado ('ct_Disponible = ct_Existencia - ct_Reservado'). Es decir, es la cantidad que realmente se podría comprometer a nuevos pedidos. La existencia incluye unidades que ya están comprometidas; el disponible es la fracción libre para vender.

**3. ¿Qué significado tendría para el negocio encontrar existencia > 0 y ventas de últimos 12 meses = 0?**

Es **capital de trabajo inmovilizado**: producto que ocupa espacio en almacén, genera costos (almacenaje, seguro, riesgo de obsolescencia) y no genera ingresos. Representa una oportunidad de acción de negocio: revisar si se trata de refacciones obsoletas o descontinuadas, aplicar descuentos o promociones para liquidarlo, redistribuirlo a otra sucursal donde sí rote, o en el peor caso darlo de baja. En este dataset se identificaron 265 productos en esa condición.

**4. Si ct_Disponible < 0, ¿lo corregirías automáticamente a 0? ¿Por qué?**

No, no lo corregiría automáticamente. Un disponible negativo es una **señal de inconsistencia que hay que investigar**, no un dato que se ajusta a ojo. Puede indicar que hay más unidades reservadas que existencias físicas (posible sobreventa, error en el registro de reservas, o existencias mal capturadas). Si lo pongo en 0 sin más, **oculto el síntoma** y el problema de fondo seguirá ocurriendo. Lo correcto es marcarlo como registro para revisión, validar contra el sistema de origen y, una vez identificada la causa, corregir en la fuente o documentar la regla de negocio aplicada. Aquí encontramos 461 productos en esta situación, lo que justifica una revisión sistematizada, no un parche puntual.

**5. ¿Qué indicador te parece más útil para responder “qué productos rotan y cuáles representan una oportunidad”?**

El cruce de **existencia > 0 con ct_VentasUltimos12Meses = 0** (inventario sin movimiento). Este indicador responde directamente a la pregunta de negocio: separa el inventario que rota del que está inmovilizado, cuantifica el capital atrapado y es directamente accionable (liquidar, mover, descontinuar). El disponible negativo, en cambio, es un indicador de calidad del dato, no de rotación. Como paso siguiente, extenderlo a una medida como *meses de cobertura* (existencia / venta promedio mensual) permitiría priorizar no solo los que no se mueven, sino los que se moverán muy lento.

## ¿Cómo se lee el trabajo ralizado hasta este punto?

describe()
    ↓
¿Cómo se comportan generalmente mis datos?

Q1 / Mediana / Q3
    ↓
¿Dónde está concentrada la mayor parte?

IQR = Q3 - Q1
    ↓
¿Cuánto ocupa el 50% central?

Límites IQR
    ↓
¿Qué valores están excepcionalmente alejados?

Outliers
    ↓
¿Por qué ocurrió? La última pregunta ya no la responde Python.
La respondes tú con el contexto de negocio.